# NYC TLC Yellow Taxi: Zone Trip Duration Analysis
**FDE Data Foundations Assignment 2 | Track B - NYC TLC Yellow Taxi**

This notebook presents an end-to-end exploratory data analysis, validation audit, spatial enrichment, and operational KPI analysis of New York City yellow taxicab trips for **June 2026** (~3.84 million raw records).


## 1. Business Question & KPI

### Business Question
> **Which NYC yellow-taxi pickup zones consistently produce longer-than-baseline trip durations, and how severe is the difference relative to the citywide non-airport baseline?**

### Intended Stakeholders
* **NYC Taxi & Limousine Commission (TLC) - Operations & Planning Team**: Needs an objective, data-backed prioritization index to flag geographic pickup zones experiencing abnormal delays.
* **TLC Analytics & Data Governance Team**: Requires formal verification that monthly vendor feeds meet strict quality, timestamp, and spatial attribution gates.

### Primary Project KPI: Zone Delay Index
$$\text{Zone Delay Index} = \frac{\text{Zone Median Trip Duration (min)}}{\text{Citywide Non-Airport Median Trip Duration (min)}}$$

* A value of **1.0** indicates that the zone's median trip duration aligns with the citywide urban norm.
* A value of **2.0** indicates that trips originating in that zone require twice as long to reach their destinations.
* **Airport trips (JFK, LaGuardia, Newark)** are excluded from the baseline and rankings because they represent fixed long-haul routes that would distort urban traffic baselines.

### Decision Supported
Identifies pickup zones that warrant targeted operational investigation based on unusually high trip-duration delay relative to the citywide baseline, without asserting unverified causal claims.


## 2. Data Sources and Loading

We utilize two primary data sources:
1. **NYC TLC Yellow Trip Data (`yellow_tripdata_2026-06.parquet`)**: High-volume trip-level records capturing timestamps, locations, distances, and meter charges.
2. **Taxi Zone Lookup (`taxi_zone_lookup.csv`)**: Reference table mapping 265 `LocationID` codes to Boroughs, Zone names, and Service Zones.


In [17]:
import pandas as pd

In [18]:
taxi_zones_df = pd.read_csv('/content/taxi_zone_lookup.csv')

print('First 10 rows of taxi_zones_df:')
display(taxi_zones_df.head(10))

print('\nSummary of taxi_zones_df:')
display(taxi_zones_df.info())

First 10 rows of taxi_zones_df:


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone



Summary of taxi_zones_df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   LocationID    265 non-null    int64 
 1   Borough       264 non-null    object
 2   Zone          264 non-null    object
 3   service_zone  263 non-null    object
dtypes: int64(1), object(3)
memory usage: 8.4+ KB


None

In [19]:
yellow_tripdata_df = pd.read_parquet('/content/yellow_tripdata_2026-06.parquet')

print('First 10 rows of yellow_tripdata_df:')
display(yellow_tripdata_df.head(10))

print('\nSummary of yellow_tripdata_df:')
display(yellow_tripdata_df.info())

First 10 rows of yellow_tripdata_df:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-06-01 00:04:22,2026-06-01 00:20:19,1.0,2.90,1.0,N,140,238,1,...,3.50,0.5,6.40,0.00,1.0,27.70,2.5,0.0,0.00,None
1,2,2026-06-01 00:59:50,2026-06-01 01:15:49,1.0,2.31,1.0,N,138,129,1,...,6.00,0.5,7.14,0.00,1.0,32.94,0.0,2.0,0.00,None
2,2,2026-06-01 00:19:05,2026-06-01 00:41:27,1.0,9.52,1.0,N,138,90,1,...,6.00,0.5,5.00,7.46,1.0,64.61,2.5,2.0,0.75,None
3,1,2026-06-01 00:24:37,2026-06-01 00:29:32,0.0,0.90,1.0,N,246,68,1,...,4.25,0.5,2.55,0.00,1.0,15.50,2.5,0.0,0.75,None
4,1,2026-06-01 00:47:37,2026-06-01 01:00:09,0.0,2.50,1.0,N,246,79,1,...,4.25,0.5,4.10,0.00,1.0,24.75,2.5,0.0,0.75,None
5,2,2026-06-01 00:56:19,2026-06-01 00:58:43,1.0,0.90,1.0,N,141,262,1,...,1.00,0.5,2.16,0.00,1.0,12.96,2.5,0.0,0.00,None
6,1,2026-06-01 00:00:42,2026-06-01 00:13:03,1.0,2.90,1.0,N,238,116,1,...,1.00,0.5,2.00,0.00,1.0,20.80,0.0,0.0,0.00,None
7,1,2026-06-01 00:30:49,2026-06-01 00:40:50,1.0,1.80,1.0,N,238,236,1,...,3.50,0.5,3.40,0.00,1.0,20.50,2.5,0.0,0.00,None
8,1,2026-06-01 00:57:38,2026-06-01 01:16:29,1.0,4.40,1.0,N,249,255,1,...,4.25,0.5,5.65,0.00,1.0,34.00,2.5,0.0,0.75,None
9,2,2026-06-01 00:03:24,2026-06-01 00:08:59,1.0,1.25,1.0,N,79,107,1,...,1.00,0.5,2.73,0.00,1.0,16.38,2.5,0.0,0.75,None



Summary of yellow_tripdata_df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3837248 entries, 0 to 3837247
Data columns (total 21 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64   

None

## 3. Initial Data Profile

Here we examine the baseline structure, memory footprint, column data types, missing values, duplicate records, and temporal range of both datasets, followed by deriving the trip duration in minutes.


In [20]:
import pandas as pd
import numpy as np

# Basic shape
print("Trip dataset shape:", yellow_tripdata_df.shape)
print("Taxi zone shape:", taxi_zones_df.shape)

# Column information
print("\nTrip dataset info:")
yellow_tripdata_df.info()

print("\nTaxi zone info:")
taxi_zones_df.info()

# Missing values
print("\nMissing values in trip dataset:")
print(yellow_tripdata_df.isna().sum().sort_values(ascending=False))

print("\nMissing values in taxi zone lookup:")
print(taxi_zones_df.isna().sum().sort_values(ascending=False))

# Duplicate rows
print("\nDuplicate trip rows:", yellow_tripdata_df.duplicated().sum())
print("Duplicate taxi zone rows:", taxi_zones_df.duplicated().sum())

# Date range
print("\nPickup time range:")
print(yellow_tripdata_df["tpep_pickup_datetime"].min())
print("to")
print(yellow_tripdata_df["tpep_pickup_datetime"].max())

print("\nDropoff time range:")
print(yellow_tripdata_df["tpep_dropoff_datetime"].min())
print("to")
print(yellow_tripdata_df["tpep_dropoff_datetime"].max())

# Numeric summary
print("\nNumeric summary:")
display(yellow_tripdata_df.describe().T)

Trip dataset shape: (3837248, 21)
Taxi zone shape: (265, 4)

Trip dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3837248 entries, 0 to 3837247
Data columns (total 21 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float

,count,mean,min,25%,50%,75%,max,std
VendorID,3837248.0,1.913787,1.0,2.0,2.0,2.0,7.0,0.714971
tpep_pickup_datetime,3837248,2026-06-15 14:45:32.925935,2008-12-31 23:03:25,2026-06-08 04:26:37.250000,2026-06-15 06:13:20,2026-06-22 23:09:03,2026-06-30 23:59:59,NaN
tpep_dropoff_datetime,3837248,2026-06-15 15:03:06.745354,2009-01-01 14:28:05,2026-06-08 04:42:45,2026-06-15 06:29:21.500000,2026-06-22 23:24:55.250000,2026-07-01 22:33:25,NaN
passenger_count,2823748.0,1.25703,0.0,1.0,1.0,1.0,9.0,0.656134
trip_distance,3837248.0,5.201764,0.0,1.01,1.82,3.64,311565.39,532.853697
RatecodeID,2823748.0,1.139986,1.0,1.0,1.0,1.0,99.0,1.992799
PULocationID,3837248.0,162.378068,1.0,116.0,161.0,234.0,265.0,66.661413
DOLocationID,3837248.0,162.310679,1.0,113.0,162.0,234.0,265.0,70.420225
payment_type,3837248.0,0.852061,0.0,0.0,1.0,1.0,5.0,0.631232
fare_amount,3837248.0,21.439724,-1683.7,10.0,15.73,26.21,7045.0,19.175335


In [21]:
# Calculate trip duration in minutes
yellow_tripdata_df["trip_duration_min"] = (
    yellow_tripdata_df["tpep_dropoff_datetime"]
    - yellow_tripdata_df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

print("Trip duration summary:")
display(yellow_tripdata_df["trip_duration_min"].describe())

print("\nTrips with non-positive duration:")
print((yellow_tripdata_df["trip_duration_min"] <= 0).sum())

print("\nTrips with zero/negative distance:")
print((yellow_tripdata_df["trip_distance"] <= 0).sum())

print("\nTrips with zero/negative total amount:")
print((yellow_tripdata_df["total_amount"] <= 0).sum())

Trip duration summary:


,trip_duration_min
count,3.837248e+06
mean,1.756366e+01
std,2.509310e+01
min,-1.778333e+01
25%,8.433333e+00
50%,1.410000e+01
75%,2.226667e+01
max,9.182433e+03



Trips with non-positive duration:
49807

Trips with zero/negative distance:
128106

Trips with zero/negative total amount:
14928


## 4. Data Quality Assessment

Before applying validation rules, we conduct targeted profiling to detect data quality anomalies:
1. Pickups falling outside June 2026 (boundary records).
2. Drop-offs occurring on or after July 1.
3. Vendor-specific missingness across optional surcharge and passenger fields.
4. Extreme physical distances (> 100 miles).
5. Negative financial totals (disputes, reversals, refund adjustments).


In [22]:
# 1. How many pickups fall outside June 2026?

pickup_start = pd.Timestamp("2026-06-01")
pickup_end = pd.Timestamp("2026-07-01")

outside_pickup = (
    (yellow_tripdata_df["tpep_pickup_datetime"] < pickup_start) |
    (yellow_tripdata_df["tpep_pickup_datetime"] >= pickup_end)
)

print("Pickups outside June 2026:", outside_pickup.sum())

print("\nOutside-date pickup records:")
display(
    yellow_tripdata_df.loc[
        outside_pickup,
        ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PULocationID", "DOLocationID", "trip_distance",
         "fare_amount", "total_amount", "request_source"]
    ].head(20)
)


# 2. How many drop-offs occur after June?

dropoff_after_june = yellow_tripdata_df["tpep_dropoff_datetime"] >= pickup_end

print("\nDrop-offs on/after July 1:", dropoff_after_june.sum())

print("\nDrop-off dates on/after July 1:")
display(
    yellow_tripdata_df.loc[
        dropoff_after_june,
        ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
         "trip_distance", "total_amount"]
    ].head(20)
)


# 3. Missingness by VendorID

missing_cols = [
    "request_source",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "RatecodeID",
    "passenger_count",
    "Airport_fee"
]

vendor_missingness = (
    yellow_tripdata_df
    .groupby("VendorID")[missing_cols]
    .apply(lambda x: x.isna().sum())
)

print("\nMissing values by VendorID:")
display(vendor_missingness)


# 4. Number of records by VendorID

print("\nRecords by VendorID:")
display(yellow_tripdata_df["VendorID"].value_counts().sort_index())


# 5. Request source distribution

print("\nRequest source distribution:")
display(yellow_tripdata_df["request_source"].value_counts(dropna=False))


# 6. Extreme distance records

print("\nTrips with distance > 100 miles:")
print((yellow_tripdata_df["trip_distance"] > 100).sum())

display(
    yellow_tripdata_df.loc[
        yellow_tripdata_df["trip_distance"] > 100,
        ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PULocationID", "DOLocationID", "trip_distance",
         "fare_amount", "total_amount"]
    ].sort_values("trip_distance", ascending=False).head(20)
)


# 7. Negative financial records

negative_financial = yellow_tripdata_df["total_amount"] < 0

print("\nTrips with negative total_amount:", negative_financial.sum())

display(
    yellow_tripdata_df.loc[
        negative_financial,
        ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
         "payment_type", "fare_amount", "tip_amount",
         "tolls_amount", "total_amount"]
    ].head(20)
)

Pickups outside June 2026: 17

Outside-date pickup records:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,total_amount,request_source
771,2,2026-05-31 23:46:19,2026-05-31 23:52:24,163,239,1.07,8.6,17.22,None
1533010,2,2026-04-16 18:12:39,2026-04-16 18:59:19,132,262,19.70,70.0,110.58,None
1558420,2,2026-04-16 21:57:54,2026-04-16 22:09:31,170,141,2.34,13.5,23.10,None
1611119,2,2008-12-31 23:03:25,2009-01-01 14:28:05,68,230,1.55,22.6,31.35,None
1624912,2,2026-04-17 17:00:18,2026-04-17 18:07:59,132,40,13.94,73.0,98.75,None
1631664,2,2026-04-17 18:21:55,2026-04-17 18:26:50,52,33,0.57,6.5,12.60,None
1638714,2,2026-04-17 19:48:59,2026-04-17 19:52:40,209,45,0.79,-5.8,-13.05,None
1638715,2,2026-04-17 19:48:59,2026-04-17 19:52:40,209,45,0.79,5.8,13.05,None
1652466,2,2026-04-17 21:36:04,2026-04-17 21:50:30,137,211,1.99,14.2,23.94,None
1658928,2,2026-04-17 22:16:58,2026-04-17 22:33:56,114,170,2.28,16.3,25.36,None



Drop-offs on/after July 1: 898

Drop-off dates on/after July 1:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,total_amount
2747470,2,2026-06-30 09:01:25,2026-07-01 08:30:28,0.87,18.25
2754402,2,2026-06-30 10:54:45,2026-07-01 10:49:51,1.55,24.42
2754471,2,2026-06-30 11:18:09,2026-07-01 10:59:47,11.57,78.76
2757778,2,2026-06-30 11:11:08,2026-07-01 10:54:19,5.80,45.14
2758447,2,2026-06-30 11:42:24,2026-07-01 11:04:21,4.69,41.95
2763922,2,2026-06-30 12:12:35,2026-07-01 12:06:31,1.18,19.65
2768999,2,2026-06-30 13:32:48,2026-07-01 13:25:42,0.70,13.35
2773387,2,2026-06-30 14:11:38,2026-07-01 13:57:16,1.06,21.90
2773772,2,2026-06-30 14:20:58,2026-07-01 14:07:14,5.36,47.10
2777742,2,2026-06-30 15:46:52,2026-07-01 15:11:50,3.06,30.60



Missing values by VendorID:


...

,count
request_source,
None,2824068
HV0003,927698
A,76711
EH0004,8107
CC,664



Trips with distance > 100 miles:
149


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,total_amount
3082237,2,2026-06-07 23:13:00,2026-06-07 23:19:00,144,107,311565.39,19.23,23.98
3470987,2,2026-06-19 03:52:00,2026-06-19 04:13:00,164,152,248207.05,30.21,34.96
2884702,2,2026-06-03 12:32:00,2026-06-03 13:05:00,68,142,221472.62,35.70,39.70
3074190,2,2026-06-07 18:53:00,2026-06-07 19:24:00,4,37,213708.86,23.18,30.72
3815493,2,2026-06-30 05:58:00,2026-06-30 06:10:00,262,161,210120.56,13.21,17.96
3394456,2,2026-06-17 05:59:00,2026-06-17 06:24:00,263,234,208137.72,23.20,27.95
3547570,2,2026-06-21 06:54:00,2026-06-21 07:39:00,116,123,197166.29,77.25,98.41
3152253,2,2026-06-10 16:35:00,2026-06-10 16:57:00,229,164,188241.40,21.20,25.95
3671318,2,2026-06-25 18:51:00,2026-06-25 19:02:00,231,249,182280.21,15.43,24.23
3368058,2,2026-06-16 05:07:00,2026-06-16 05:16:00,231,246,179492.47,18.78,28.24



Trips with negative total_amount: 14312


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,payment_type,fare_amount,tip_amount,tolls_amount,total_amount
83,2,2026-06-01 00:52:23,2026-06-01 01:16:59,4,-46.4,0.0,0.00,-50.90
130,2,2026-06-01 00:00:29,2026-06-01 00:12:31,4,-14.2,0.0,0.00,-19.95
429,2,2026-06-01 00:56:52,2026-06-01 01:48:50,4,-73.7,0.0,0.00,-83.20
783,2,2026-06-01 00:42:40,2026-06-01 00:59:46,4,-20.5,0.0,0.00,-25.50
806,2,2026-06-01 00:21:38,2026-06-01 00:22:18,2,-3.0,0.0,0.00,-8.00
846,2,2026-06-01 00:48:06,2026-06-01 19:56:03,2,0.0,0.0,0.00,-1.50
859,2,2026-06-01 00:51:14,2026-06-01 01:10:01,4,-37.3,0.0,0.00,-41.80
931,2,2026-06-01 00:50:48,2026-06-01 00:52:24,2,-70.0,0.0,0.00,-74.75
1062,2,2026-06-01 00:29:37,2026-06-01 00:59:53,4,-69.5,0.0,-7.46,-81.46
1171,2,2026-06-01 01:11:51,2026-06-01 01:15:47,2,-5.8,0.0,0.00,-11.55


In [38]:
missing_audit = (
    yellow_tripdata_df.isna()
    .sum()
    .reset_index()
)

missing_audit.columns = ["Column", "Missing_Rows"]

missing_audit["Missing_Percentage"] = (
    missing_audit["Missing_Rows"] / raw_rows * 100
).round(2)

missing_audit = missing_audit[
    missing_audit["Missing_Rows"] > 0
].sort_values("Missing_Rows", ascending=False)

display(missing_audit)

,Column,Missing_Rows,Missing_Percentage
20,request_source,2824068,73.60
3,passenger_count,1013500,26.41
5,RatecodeID,1013500,26.41
6,store_and_fwd_flag,1013500,26.41
17,congestion_surcharge,1013500,26.41
18,Airport_fee,1013500,26.41


## 5. Validation Rules

We establish five domain-specific business validation rules. Each rule is stored as an auditable boolean column so no records are silently dropped without tracking:
1. `valid_pickup_date`: Pickup timestamp falls strictly within `[2026-06-01, 2026-07-01)`.
2. `valid_timestamp_order`: Drop-off timestamp is strictly greater than pickup timestamp.
3. `valid_distance`: Trip distance is strictly greater than 0 miles.
4. `valid_pickup_zone`: `PULocationID` exists in the authoritative TLC zone lookup table.
5. `valid_dropoff_zone`: `DOLocationID` exists in the authoritative TLC zone lookup table.

An overall gate `is_valid` requires all five rules to evaluate to `True`.


In [23]:
# Validation flags

yellow_tripdata_df["valid_pickup_date"] = (
    (yellow_tripdata_df["tpep_pickup_datetime"] >= "2026-06-01") &
    (yellow_tripdata_df["tpep_pickup_datetime"] < "2026-07-01")
)

yellow_tripdata_df["valid_timestamp_order"] = (
    yellow_tripdata_df["tpep_dropoff_datetime"] >
    yellow_tripdata_df["tpep_pickup_datetime"]
)

yellow_tripdata_df["valid_distance"] = (
    yellow_tripdata_df["trip_distance"] > 0
)

yellow_tripdata_df["valid_duration"] = (
    yellow_tripdata_df["trip_duration_min"] > 0
)

yellow_tripdata_df["valid_pickup_zone"] = (
    yellow_tripdata_df["PULocationID"].isin(taxi_zones_df["LocationID"])
)

yellow_tripdata_df["valid_dropoff_zone"] = (
    yellow_tripdata_df["DOLocationID"].isin(taxi_zones_df["LocationID"])
)

# Overall validation status
yellow_tripdata_df["is_valid"] = (
    yellow_tripdata_df["valid_pickup_date"] &
    yellow_tripdata_df["valid_timestamp_order"] &
    yellow_tripdata_df["valid_distance"] &
    yellow_tripdata_df["valid_duration"] &
    yellow_tripdata_df["valid_pickup_zone"] &
    yellow_tripdata_df["valid_dropoff_zone"]
)

# Summary
validation_summary = pd.DataFrame({
    "Check": [
        "Pickup date in June",
        "Dropoff after pickup",
        "Positive distance",
        "Positive duration",
        "Valid pickup zone",
        "Valid dropoff zone",
        "Overall valid"
    ],
    "Invalid_Count": [
        (~yellow_tripdata_df["valid_pickup_date"]).sum(),
        (~yellow_tripdata_df["valid_timestamp_order"]).sum(),
        (~yellow_tripdata_df["valid_distance"]).sum(),
        (~yellow_tripdata_df["valid_duration"]).sum(),
        (~yellow_tripdata_df["valid_pickup_zone"]).sum(),
        (~yellow_tripdata_df["valid_dropoff_zone"]).sum(),
        (~yellow_tripdata_df["is_valid"]).sum()
    ]
})

display(validation_summary)

print(
    "Valid records:",
    yellow_tripdata_df["is_valid"].sum()
)

print(
    "Valid %:",
    round(yellow_tripdata_df["is_valid"].mean() * 100, 2)
)

,Check,Invalid_Count
0,Pickup date in June,17
1,Dropoff after pickup,49807
2,Positive distance,128106
3,Positive duration,49807
4,Valid pickup zone,0
5,Valid dropoff zone,0
6,Overall valid,176773


Valid records: 3660475
Valid %: 95.39


## 6. Data Cleaning and Analytical Dataset

We partition the records into:
1. **Trusted Trips**: Records passing all five core validation rules (`is_valid == True`).
2. **Secondary Analytical Filter (`total_amount > 0`)**: Excludes 12,932 zero or negative charge records representing credit adjustments or meter cancellations, isolating genuine completed passenger journeys.


In [24]:
trusted_trips_df = yellow_tripdata_df[
    yellow_tripdata_df["is_valid"]
].copy()

print("Trusted dataset shape:", trusted_trips_df.shape)

Trusted dataset shape: (3660475, 29)


In [25]:
trusted_trips_df["pickup_hour"] = (
    trusted_trips_df["tpep_pickup_datetime"].dt.hour
)

trusted_trips_df["trip_duration_min"] = (
    trusted_trips_df["tpep_dropoff_datetime"] -
    trusted_trips_df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

trusted_trips_df["revenue_per_hour"] = (
    trusted_trips_df["total_amount"] /
    (trusted_trips_df["trip_duration_min"] / 60)
)

trusted_trips_df[[
    "trip_duration_min",
    "trip_distance",
    "total_amount",
    "revenue_per_hour"
]].describe()

,trip_duration_min,trip_distance,total_amount,revenue_per_hour
count,3.660475e+06,3.660475e+06,3.660475e+06,3.660475e+06
mean,1.787613e+01,5.416950e+00,3.039996e+01,2.007086e+02
std,2.491735e+01,5.455672e+02,2.250060e+01,2.423030e+03
min,1.666667e-02,1.000000e-02,-1.815180e+03,-3.156000e+05
25%,8.700000e+00,1.100000e+00,1.755000e+01,8.768489e+01
50%,1.426667e+01,1.900000e+00,2.375000e+01,1.066397e+02
75%,2.241667e+01,3.790000e+00,3.475000e+01,1.352511e+02
max,9.182433e+03,3.115654e+05,1.830100e+03,4.812000e+05


In [26]:
print("Negative total amount:")
print((trusted_trips_df["total_amount"] < 0).sum())

print("\nZero total amount:")
print((trusted_trips_df["total_amount"] == 0).sum())

print("\nExtreme trip distance (>100 miles):")
print((trusted_trips_df["trip_distance"] > 100).sum())

print("\nExtreme duration (>3 hours):")
print((trusted_trips_df["trip_duration_min"] > 180).sum())

Negative total amount:
12544

Zero total amount:
388

Extreme trip distance (>100 miles):
148

Extreme duration (>3 hours):
1074


In [27]:
kpi_trips_df = trusted_trips_df[
    trusted_trips_df["total_amount"] > 0
].copy()

print("KPI dataset shape:", kpi_trips_df.shape)

kpi_trips_df["revenue_per_hour"] = (
    kpi_trips_df["total_amount"] /
    (kpi_trips_df["trip_duration_min"] / 60)
)

kpi_trips_df["revenue_per_mile"] = (
    kpi_trips_df["total_amount"] /
    kpi_trips_df["trip_distance"]
)

display(
    kpi_trips_df[
        ["trip_duration_min", "trip_distance",
         "total_amount", "revenue_per_hour", "revenue_per_mile"]
    ].describe()
)

KPI dataset shape: (3647543, 31)


,trip_duration_min,trip_distance,total_amount,revenue_per_hour,revenue_per_mile
count,3.647543e+06,3.647543e+06,3.647543e+06,3.647543e+06,3.647543e+06
mean,1.787550e+01,5.419916e+00,3.061094e+01,2.039943e+02,3.598496e+01
std,2.487822e+01,5.465178e+02,2.213329e+01,2.403892e+03,3.151714e+02
min,1.666667e-02,1.000000e-02,5.000000e-01,7.840063e-02,7.696619e-05
25%,8.716667e+00,1.100000e+00,1.763000e+01,8.787947e+01,8.416667e+00
50%,1.428333e+01,1.900000e+00,2.385000e+01,1.067904e+02,1.210000e+01
75%,2.241667e+01,3.780000e+00,3.485000e+01,1.353936e+02,1.707317e+01
max,9.182433e+03,3.115654e+05,1.830100e+03,4.812000e+05,5.810000e+04


In [35]:
clean_trips_df = trusted_trips_df[
    trusted_trips_df["total_amount"] > 0
].copy()

print("Raw rows:", len(yellow_tripdata_df))
print("After validation:", len(trusted_trips_df))
print("Final analytical rows:", len(clean_trips_df))
print("Rows removed:", len(yellow_tripdata_df) - len(clean_trips_df))
print("Final retention:", round(len(clean_trips_df) / len(yellow_tripdata_df) * 100, 2), "%")


Raw rows: 3837248
After validation: 3660475
Final analytical rows: 3647543
Rows removed: 189705
Final retention: 95.06 %


In [36]:
cleaning_audit = pd.DataFrame({
    "Stage": [
        "Raw TLC data",
        "After core validation",
        "Final analytical dataset"
    ],
    "Rows": [
        len(yellow_tripdata_df),
        len(trusted_trips_df),
        len(clean_trips_df)
    ]
})

display(cleaning_audit)

,Stage,Rows
0,Raw TLC data,3837248
1,After core validation,3660475
2,Final analytical dataset,3647543


In [37]:
# Final data-quality audit

raw_rows = len(yellow_tripdata_df)

quality_audit = pd.DataFrame({
    "Check": [
        "Pickup outside June 2026",
        "Dropoff before/equal pickup",
        "Non-positive distance",
        "Invalid pickup zone",
        "Invalid dropoff zone",
        "Non-positive total amount",
        "Duplicate rows"
    ],
    "Affected_Rows": [
        (~yellow_tripdata_df["valid_pickup_date"]).sum(),
        (~yellow_tripdata_df["valid_timestamp_order"]).sum(),
        (~yellow_tripdata_df["valid_distance"]).sum(),
        (~yellow_tripdata_df["valid_pickup_zone"]).sum(),
        (~yellow_tripdata_df["valid_dropoff_zone"]).sum(),
        (yellow_tripdata_df["total_amount"] <= 0).sum(),
        yellow_tripdata_df.duplicated().sum()
    ]
})

quality_audit["Percentage"] = (
    quality_audit["Affected_Rows"] / raw_rows * 100
).round(3)

display(quality_audit)

,Check,Affected_Rows,Percentage
0,Pickup outside June 2026,17,0.000
1,Dropoff before/equal pickup,49807,1.298
2,Non-positive distance,128106,3.338
3,Invalid pickup zone,0,0.000
4,Invalid dropoff zone,0,0.000
5,Non-positive total amount,14928,0.389
6,Duplicate rows,0,0.000


## 7. Zone Enrichment

To calculate zone-level KPIs, we left-join spatial metadata (Borough, Zone, service_zone) for both pickup and drop-off locations. 

Special considerations:
* `LocationID` values 264 and 265 correspond to "Unknown / NA". Rather than dropping these trips, we preserve them and explicitly populate their zone fields as `"Unknown"`.
* We verify that the join preserves exact 1:1 row cardinality and introduces zero duplicate rows.


In [39]:
# Final analytical dataset with pickup + drop-off zone details

pu_zones = taxi_zones_df.rename(columns={
    "LocationID": "PULocationID",
    "Borough": "PU_Borough",
    "Zone": "PU_Zone",
    "service_zone": "PU_service_zone"
})

do_zones = taxi_zones_df.rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "DO_Borough",
    "Zone": "DO_Zone",
    "service_zone": "DO_service_zone"
})

final_trips_df = clean_trips_df.merge(
    pu_zones,
    on="PULocationID",
    how="left"
).merge(
    do_zones,
    on="DOLocationID",
    how="left"
)

print("Final analytical dataset:", final_trips_df.shape)

display(
    final_trips_df[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "trip_duration_min",
            "PULocationID",
            "PU_Borough",
            "PU_Zone",
            "DOLocationID",
            "DO_Borough",
            "DO_Zone",
            "total_amount"
        ]
    ].head()
)


Final analytical dataset: (3647543, 37)


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,trip_duration_min,PULocationID,PU_Borough,PU_Zone,DOLocationID,DO_Borough,DO_Zone,total_amount
0,2026-06-01 00:04:22,2026-06-01 00:20:19,2.90,15.950000,140,Manhattan,Lenox Hill East,238,Manhattan,Upper West Side North,27.70
1,2026-06-01 00:59:50,2026-06-01 01:15:49,2.31,15.983333,138,Queens,LaGuardia Airport,129,Queens,Jackson Heights,32.94
2,2026-06-01 00:19:05,2026-06-01 00:41:27,9.52,22.366667,138,Queens,LaGuardia Airport,90,Manhattan,Flatiron,64.61
3,2026-06-01 00:24:37,2026-06-01 00:29:32,0.90,4.916667,246,Manhattan,West Chelsea/Hudson Yards,68,Manhattan,East Chelsea,15.50
4,2026-06-01 00:47:37,2026-06-01 01:00:09,2.50,12.533333,246,Manhattan,West Chelsea/Hudson Yards,79,Manhattan,East Village,24.75


In [40]:
print("Missing pickup zones after join:",
      final_trips_df["PU_Zone"].isna().sum())

print("Missing drop-off zones after join:",
      final_trips_df["DO_Zone"].isna().sum())

Missing pickup zones after join: 3654
Missing drop-off zones after join: 3389


In [41]:
print("Pickup IDs with missing zone name:")
display(
    final_trips_df.loc[
        final_trips_df["PU_Zone"].isna(),
        "PULocationID"
    ].value_counts()
)

print("\nDrop-off IDs with missing zone name:")
display(
    final_trips_df.loc[
        final_trips_df["DO_Zone"].isna(),
        "DOLocationID"
    ].value_counts()
)


Pickup IDs with missing zone name:


,count
PULocationID,
264,3654



Drop-off IDs with missing zone name:


,count
DOLocationID,
264,3389


In [42]:
# Preserve the trips, but make missing zone metadata explicit

zone_columns = [
    "PU_Borough", "PU_Zone", "PU_service_zone",
    "DO_Borough", "DO_Zone", "DO_service_zone"
]

final_trips_df[zone_columns] = (
    final_trips_df[zone_columns].fillna("Unknown")
)

print("Unknown pickup zones:",
      (final_trips_df["PU_Zone"] == "Unknown").sum())

print("Unknown drop-off zones:",
      (final_trips_df["DO_Zone"] == "Unknown").sum())

Unknown pickup zones: 3654
Unknown drop-off zones: 3389


### Cleaning and Validation Decisions

Records were excluded when:
- pickup date was outside June 2026
- drop-off occurred before or at pickup
- trip distance was non-positive
- total amount was non-positive

Records were retained when they contained extreme but not inherently invalid
distance or duration values, because these require investigation rather than
an arbitrary cutoff.

Missing source fields such as request_source, passenger_count, RatecodeID,
and Airport_fee were not imputed.

Trips with LocationID 264 were retained and their missing zone metadata was
labelled as "Unknown".

In [43]:
print("Rows before zone join:", len(clean_trips_df))
print("Rows after zone join:", len(final_trips_df))
print("Duplicate rows after join:", final_trips_df.duplicated().sum())

assert len(clean_trips_df) == len(final_trips_df)
assert final_trips_df.duplicated().sum() == 0

print("Final integrity check: PASSED")

Rows before zone join: 3647543
Rows after zone join: 3647543
Duplicate rows after join: 0
Final integrity check: PASSED


## 8. KPI Analysis

Having validated, filtered, and spatially enriched the trip data, we now compute the project's analytical metrics and primary KPI.


### 8.1 Citywide Baseline (Airport vs Non-Airport)

Trips to and from major airports (JFK, LaGuardia, Newark) have fixed routes and significantly longer durations. We isolate airport pickups to establish an accurate citywide non-airport median baseline (13.60 min).


In [30]:
airport_zones = ["JFK Airport", "LaGuardia Airport", "Newark Airport"]

kpi_trips_df = kpi_trips_df.merge(
    taxi_zones_df[["LocationID", "Zone"]],
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

kpi_trips_df["is_airport_pickup"] = (
    kpi_trips_df["Zone"].isin(airport_zones)
)

airport_comparison = (
    kpi_trips_df
    .groupby("is_airport_pickup")
    .agg(
        trips=("VendorID", "size"),
        median_duration=("trip_duration_min", "median"),
        p90_duration=("trip_duration_min", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

display(airport_comparison)

,is_airport_pickup,trips,median_duration,p90_duration
0,False,3407251,13.60,29.35
1,True,240292,37.95,66.75


In [31]:
non_airport = kpi_trips_df[
    ~kpi_trips_df["is_airport_pickup"]
].copy()

non_airport_zone_metrics = (
    non_airport
    .groupby("PULocationID")
    .agg(
        trips=("VendorID", "size"),
        median_duration=("trip_duration_min", "median"),
        p90_duration=("trip_duration_min", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

non_airport_zone_metrics = non_airport_zone_metrics[
    non_airport_zone_metrics["trips"] >= 1000
].copy()

non_airport_zone_metrics = non_airport_zone_metrics.merge(
    taxi_zones_df[["LocationID", "Borough", "Zone"]],
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

non_airport_zone_metrics = non_airport_zone_metrics.sort_values(
    "median_duration", ascending=False
)

display(
    non_airport_zone_metrics[
        ["PULocationID", "Borough", "Zone",
         "trips", "median_duration", "p90_duration"]
    ].head(15)
)

,PULocationID,Borough,Zone,trips,median_duration,p90_duration
2,10,Queens,Baisley Park,1202,38.033333,66.300000
26,70,Queens,East Elmhurst,8196,32.133333,56.108333
35,89,Brooklyn,Flatbush/Ditmas Park,1517,26.633333,47.723333
83,225,Brooklyn,Stuyvesant Heights,2978,26.033333,42.731667
22,62,Brooklyn,Crown Heights South,1036,25.975000,45.191667
85,228,Brooklyn,Sunset Park West,1626,25.916667,41.533333
75,189,Brooklyn,Prospect Heights,1443,25.316667,42.550000
74,188,Brooklyn,Prospect-Lefferts Gardens,1831,24.983333,45.900000
21,61,Brooklyn,Crown Heights North,4076,24.491667,43.325000
72,181,Brooklyn,Park Slope,3459,24.033333,41.876667


### 8.2 Zone Delay Index

We calculate the **Zone Delay Index** for each pickup zone against the 13.60-minute non-airport baseline. 

To ensure statistical reliability and prevent small sample bias from generating false alarm delay signals, we enforce a minimum threshold of **3,000 trips** per zone.


In [29]:
overall_median = kpi_trips_df["trip_duration_min"].median()

zone_metrics = (
    kpi_trips_df
    .groupby("PULocationID")
    .agg(
        trips=("VendorID", "size"),
        median_duration=("trip_duration_min", "median"),
        p90_duration=("trip_duration_min", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

# Avoid drawing conclusions from tiny samples
zone_metrics = zone_metrics[zone_metrics["trips"] >= 1000].copy()

zone_metrics["delay_index"] = (
    zone_metrics["median_duration"] / overall_median
)

zone_metrics = zone_metrics.merge(
    taxi_zones_df[["LocationID", "Borough", "Zone"]],
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

zone_metrics = zone_metrics.sort_values(
    "delay_index",
    ascending=False
)

display(
    zone_metrics[
        ["PULocationID", "Borough", "Zone",
         "trips", "median_duration",
         "p90_duration", "delay_index"]
    ].head(15)
)

,PULocationID,Borough,Zone,trips,median_duration,p90_duration,delay_index
50,132,Queens,JFK Airport,142263,42.683333,71.550000,2.988331
2,10,Queens,Baisley Park,1202,38.033333,66.300000,2.662777
52,138,Queens,LaGuardia Airport,97897,32.866667,56.250000,2.301050
26,70,Queens,East Elmhurst,8196,32.133333,56.108333,2.249708
35,89,Brooklyn,Flatbush/Ditmas Park,1517,26.633333,47.723333,1.864644
85,225,Brooklyn,Stuyvesant Heights,2978,26.033333,42.731667,1.822637
22,62,Brooklyn,Crown Heights South,1036,25.975000,45.191667,1.818553
87,228,Brooklyn,Sunset Park West,1626,25.916667,41.533333,1.814469
77,189,Brooklyn,Prospect Heights,1443,25.316667,42.550000,1.772462
76,188,Brooklyn,Prospect-Lefferts Gardens,1831,24.983333,45.900000,1.749125


In [32]:
baseline_duration = non_airport["trip_duration_min"].median()

zone_kpi = (
    non_airport
    .groupby("PULocationID")
    .agg(
        trips=("VendorID", "size"),
        median_duration=("trip_duration_min", "median"),
        p90_duration=("trip_duration_min", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

zone_kpi = zone_kpi[zone_kpi["trips"] >= 3000].copy()

zone_kpi["delay_index"] = (
    zone_kpi["median_duration"] / baseline_duration
)

zone_kpi = zone_kpi.merge(
    taxi_zones_df[["LocationID", "Borough", "Zone"]],
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

zone_kpi = zone_kpi.sort_values("delay_index", ascending=False)

display(
    zone_kpi[
        ["Borough", "Zone", "trips",
         "median_duration", "p90_duration", "delay_index"]
    ].head(15)
)

,Borough,Zone,trips,median_duration,p90_duration,delay_index
18,Queens,East Elmhurst,8196,32.133333,56.108333,2.362745
14,Brooklyn,Crown Heights North,4076,24.491667,43.325000,1.800858
53,Brooklyn,Park Slope,3459,24.033333,41.876667,1.767157
7,Brooklyn,Bushwick South,5320,23.858333,39.100000,1.754289
6,Brooklyn,Bushwick North,3130,23.350000,36.650000,1.716912
16,Brooklyn,DUMBO/Vinegar Hill,5003,21.516667,35.546667,1.582108
5,Brooklyn,Brooklyn Heights,4818,21.425000,35.883333,1.575368
73,Brooklyn,Williamsburg (North Side),9521,21.083333,33.816667,1.550245
4,Brooklyn,Boerum Hill,3592,21.025000,36.271667,1.545956
22,Brooklyn,East Williamsburg,5132,20.875000,33.548333,1.534926


### 8.3 Highest-Delay Zones

Among the 79 qualifying zones (>= 3,000 trips), **East Elmhurst** in Queens ranks highest with a median trip duration of **32.13 minutes**, yielding a **Zone Delay Index of 2.36x** and a 90th percentile (P90) duration of **56.11 minutes**.

*Reference chart: [`outputs/top_zone_delay_index.png`](../outputs/top_zone_delay_index.png) illustrates the top 10 delay zones ranked against the 1.00x non-airport baseline.*


### 8.4 Peak Demand Hour

We aggregate trip volumes by pickup hour to identify system-wide peak operational pressure. Peak travel demand occurs at **18:00 (6:00 PM)** with **231,529 trips** in the month.

*Reference charts: [`outputs/hourly_trip_demand.png`](../outputs/hourly_trip_demand.png) depicts diurnal demand curve, and [`outputs/hourly_duration.png`](../outputs/hourly_duration.png) depicts hourly median and P90 duration trajectories.*


In [28]:
overall_median_duration = kpi_trips_df["trip_duration_min"].median()
p90_duration = kpi_trips_df["trip_duration_min"].quantile(0.90)

print("Median trip duration:", round(overall_median_duration, 2), "minutes")
print("P90 trip duration:", round(p90_duration, 2), "minutes")

kpi_trips_df["pickup_hour"] = (
    kpi_trips_df["tpep_pickup_datetime"].dt.hour
)

hourly_metrics = (
    kpi_trips_df
    .groupby("pickup_hour")
    .agg(
        trips=("VendorID", "size"),
        median_duration=("trip_duration_min", "median"),
        p90_duration=("trip_duration_min", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

display(hourly_metrics)

Median trip duration: 14.28 minutes
P90 trip duration: 33.23 minutes


,pickup_hour,trips,median_duration,p90_duration
0,0,131570,14.016667,29.000000
1,1,82971,13.333333,26.200000
2,2,55563,13.000000,25.033333
3,3,38566,12.916667,25.066667
4,4,30791,13.400000,26.766667
5,5,33091,12.716667,29.250000
6,6,59504,11.933333,30.211667
7,7,105356,12.600000,30.650000
8,8,145332,13.783333,31.700000
9,9,150449,14.100000,32.233333


## 9. Key Findings

Below are the summarized key findings and executive evidence table generated from June 2026 data:


In [33]:
final_kpis = {
    "Valid Trip Rate": round(yellow_tripdata_df["is_valid"].mean() * 100, 2),

    "Peak Demand Hour": int(
        hourly_metrics.loc[hourly_metrics["trips"].idxmax(), "pickup_hour"]
    ),

    "Peak Hour Trips": int(hourly_metrics["trips"].max()),

    "Median Trip Duration (min)": round(
        kpi_trips_df["trip_duration_min"].median(), 2
    ),

    "P90 Trip Duration (min)": round(
        kpi_trips_df["trip_duration_min"].quantile(0.90), 2
    ),

    "Airport Median Duration (min)": round(
        airport_comparison.loc[
            airport_comparison["is_airport_pickup"] == True,
            "median_duration"
        ].iloc[0], 2
    ),

    "Non-Airport Median Duration (min)": round(
        airport_comparison.loc[
            airport_comparison["is_airport_pickup"] == False,
            "median_duration"
        ].iloc[0], 2
    ),

    "Highest Delay Zone": zone_kpi.iloc[0]["Zone"],

    "Highest Zone Delay Index": round(
        zone_kpi.iloc[0]["delay_index"], 2
    )
}

for metric, value in final_kpis.items():
    print(f"{metric}: {value}")

Valid Trip Rate: 95.39
Peak Demand Hour: 18
Peak Hour Trips: 231529
Median Trip Duration (min): 14.28
P90 Trip Duration (min): 33.23
Airport Median Duration (min): 37.95
Non-Airport Median Duration (min): 13.6
Highest Delay Zone: East Elmhurst
Highest Zone Delay Index: 2.36


In [34]:
final_evidence = pd.DataFrame({
    "Metric": [
        "Valid Trip Rate",
        "Peak Demand Hour",
        "Peak Hour Trips",
        "Median Trip Duration",
        "P90 Trip Duration",
        "Airport Median Duration",
        "Non-Airport Median Duration",
        "Highest Delay Zone",
        "Highest Zone Delay Index"
    ],
    "Value": [
        "95.39%",
        "18:00",
        231529,
        "14.28 min",
        "33.23 min",
        "37.95 min",
        "13.60 min",
        "East Elmhurst",
        "2.36x"
    ]
})

display(final_evidence)

,Metric,Value
0,Valid Trip Rate,95.39%
1,Peak Demand Hour,18:00
2,Peak Hour Trips,231529
3,Median Trip Duration,14.28 min
4,P90 Trip Duration,33.23 min
5,Airport Median Duration,37.95 min
6,Non-Airport Median Duration,13.60 min
7,Highest Delay Zone,East Elmhurst
8,Highest Zone Delay Index,2.36x


### Summary of Empirical Evidence
* **Core Data Trustworthiness**: 95.39% of raw records satisfy all physical and temporal constraints.
* **Citywide Baseline**: Typical urban non-airport trips take a median of **13.60 minutes** (mean: 14.28 minutes).
* **Extreme Tail Delay**: East Elmhurst exhibits a delay index of **2.36x**, with 10% of trips taking over **56 minutes**.
* **Demand Concentration**: System-wide demand peaks during the evening commute (18:00) with over 231,000 rides.
* **Visual Evidence Artifacts**: Pre-rendered analytical charts are available in `outputs/` (`top_zone_delay_index.png`, `hourly_trip_demand.png`, `hourly_duration.png`).


## 10. Limitations and Assumptions (KUAL Framework)

### Known
* **4.61% of records fail validation**: Zero/negative distance (128,106 rows) and non-positive duration (49,807 rows) account for almost all rejections.
* **12,932 records with total_amount <= 0** were excluded as credit refunds/adjustments.
* **3,654 pickup zones remain 'Unknown'** (LocationID 264) due to unassigned TLC boundary coordinates.

### Unknown
* **Root Causes**: Data does not indicate whether delay is caused by localized surface congestion, highway construction, or circuitous routing.
* **Driver / Fleet Variation**: No driver or vehicle IDs exist in the public dataset.
* **Unmet Demand**: Passengers unable to find a cab in high-delay zones are invisible.

### Assumptions
* Positive `total_amount` is a reliable indicator of a completed revenue trip.
* The 13.60-minute non-airport median provides a representative benchmark for urban taxi travel.
* Zones with >= 3,000 trips provide sufficient statistical stability for median comparisons.

### Limitations
* **Single Month Scope**: Analysis reflects June 2026 conditions; seasonal weather and holiday variations are not captured.
* **Duration as a Proxy**: Duration conflates trip distance with traffic congestion; longer trips naturally produce higher medians.


## 11. Conclusion & Operational Recommendations

1. **Prioritize Targeted Investigation**: TLC Operations should prioritize field reviews for **East Elmhurst** (2.36x) and nearby transit corridors rather than applying citywide blanket regulations.
2. **Incorporate Real-Time Traffic Layers**: Future iterations should merge external MTA/DOT speed sensors and weather feeds to explain the root causes of observed trip duration anomalies.
3. **Automate Pipeline Ingestion**: The data governance checks established here should run as a monthly automated gate before publishing open data reports.


## Appendix: Reproducible Pipeline Execution & Verification

The cells below demonstrate the standalone, automated pipeline workflow and verify output file schemas and integrity assertions.


### FINAL REPRODUCBLE PIPELINE

In [44]:
from pathlib import Path
import pandas as pd

RAW_TRIPS = Path("/content/yellow_tripdata_2026-06.parquet")
RAW_ZONES = Path("/content/taxi_zone_lookup.csv")

OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Raw trip file exists:", RAW_TRIPS.exists())
print("Raw zone file exists:", RAW_ZONES.exists())

Raw trip file exists: True
Raw zone file exists: True


In [46]:
# =========================
# Pipeline Configuration
# =========================

YEAR = 2026
MONTH = 6

MONTH_ID = f"{YEAR}-{MONTH:02d}"

RAW_TRIPS = Path(f"/content/yellow_tripdata_{MONTH_ID}.parquet")
RAW_ZONES = Path("/content/taxi_zone_lookup.csv")

START_DATE = pd.Timestamp(YEAR, MONTH, 1)

if MONTH == 12:
    END_DATE = pd.Timestamp(YEAR + 1, 1, 1)
else:
    END_DATE = pd.Timestamp(YEAR, MONTH + 1, 1)

OUTPUT_DIR = Path(f"/content/output/{MONTH_ID}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Processing:", MONTH_ID)
print("Trip file:", RAW_TRIPS)
print("Period:", START_DATE.date(), "to", END_DATE.date())

Processing: 2026-06
Trip file: /content/yellow_tripdata_2026-06.parquet
Period: 2026-06-01 to 2026-07-01


In [47]:
from pathlib import Path
import pandas as pd

RAW_TRIPS = Path("/content/yellow_tripdata_2026-06.parquet")
RAW_ZONES = Path("/content/taxi_zone_lookup.csv")
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Ingest raw data
trips = pd.read_parquet(RAW_TRIPS)
zones = pd.read_csv(RAW_ZONES)

raw_rows = len(trips)

# Trip duration
trips["trip_duration_min"] = (
    trips["tpep_dropoff_datetime"]
    - trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Validation rules
trips["valid_pickup_date"] = (
    (trips["tpep_pickup_datetime"] >= START_DATE) &
    (trips["tpep_pickup_datetime"] < END_DATE)
)

trips["valid_timestamp_order"] = (
    trips["tpep_dropoff_datetime"] >
    trips["tpep_pickup_datetime"]
)

trips["valid_distance"] = trips["trip_distance"] > 0

trips["valid_pickup_zone"] = trips["PULocationID"].isin(
    zones["LocationID"]
)

trips["valid_dropoff_zone"] = trips["DOLocationID"].isin(
    zones["LocationID"]
)

trips["is_valid"] = (
    trips["valid_pickup_date"] &
    trips["valid_timestamp_order"] &
    trips["valid_distance"] &
    trips["valid_pickup_zone"] &
    trips["valid_dropoff_zone"]
)

validated = trips[trips["is_valid"]].copy()

# Analytical filtering
clean_trips = validated[
    validated["total_amount"] > 0
].copy()

print("Raw rows:", raw_rows)
print("After validation:", len(validated))
print("Final analytical rows:", len(clean_trips))

Raw rows: 3837248
After validation: 3660475
Final analytical rows: 3647543


In [48]:
# Zone enrichment

pu_zones = zones.rename(columns={
    "LocationID": "PULocationID",
    "Borough": "PU_Borough",
    "Zone": "PU_Zone",
    "service_zone": "PU_service_zone"
})

do_zones = zones.rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "DO_Borough",
    "Zone": "DO_Zone",
    "service_zone": "DO_service_zone"
})

final_trips = (
    clean_trips
    .merge(pu_zones, on="PULocationID", how="left")
    .merge(do_zones, on="DOLocationID", how="left")
)

# Explicitly handle missing lookup metadata
zone_cols = [
    "PU_Borough", "PU_Zone", "PU_service_zone",
    "DO_Borough", "DO_Zone", "DO_service_zone"
]

final_trips[zone_cols] = final_trips[zone_cols].fillna("Unknown")

# KPI calculations
final_trips["pickup_hour"] = final_trips["tpep_pickup_datetime"].dt.hour

hourly_metrics = (
    final_trips
    .groupby("pickup_hour")
    .agg(
        trips=("VendorID", "size"),
        median_duration=("trip_duration_min", "median"),
        p90_duration=("trip_duration_min", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

overall_median = final_trips["trip_duration_min"].median()

non_airport = final_trips[
    ~final_trips["PU_Zone"].isin(
        ["JFK Airport", "LaGuardia Airport", "Newark Airport"]
    )
]

zone_kpi = (
    non_airport
    .groupby(["PULocationID", "PU_Borough", "PU_Zone"])
    .agg(
        trips=("VendorID", "size"),
        median_duration=("trip_duration_min", "median"),
        p90_duration=("trip_duration_min", lambda x: x.quantile(0.90))
    )
    .reset_index()
)

zone_kpi = zone_kpi[zone_kpi["trips"] >= 3000].copy()

baseline = non_airport["trip_duration_min"].median()

zone_kpi["delay_index"] = (
    zone_kpi["median_duration"] / baseline
)

zone_kpi = zone_kpi.sort_values(
    "delay_index", ascending=False
)

airport = final_trips[
    final_trips["PU_Zone"].isin(
        ["JFK Airport", "LaGuardia Airport", "Newark Airport"]
    )
]

airport_median = airport["trip_duration_min"].median()
non_airport_median = non_airport["trip_duration_min"].median()

final_kpis = pd.DataFrame({
    "Metric": [
        "Core Valid Trip Rate",
        "Final Analytical Retention",
        "Peak Demand Hour",
        "Peak Hour Trips",
        "Median Trip Duration",
        "P90 Trip Duration",
        "Airport Median Duration",
        "Non-Airport Median Duration",
        "Highest Delay Zone",
        "Highest Zone Delay Index"
    ],
    "Value": [
        round(trips["is_valid"].mean() * 100, 2),
        round(len(final_trips) / raw_rows * 100, 2),
        f"{hourly_metrics.loc[hourly_metrics['trips'].idxmax(), 'pickup_hour']:02d}:00",
        int(hourly_metrics["trips"].max()),
        round(final_trips["trip_duration_min"].median(), 2),
        round(final_trips["trip_duration_min"].quantile(0.90), 2),
        round(airport_median, 2),
        round(non_airport_median, 2),
        zone_kpi.iloc[0]["PU_Zone"],
        round(zone_kpi.iloc[0]["delay_index"], 2)
    ]
})

# Save pipeline outputs
final_kpis.to_csv(OUTPUT_DIR / "final_kpis.csv", index=False)
hourly_metrics.to_csv(OUTPUT_DIR / "hourly_metrics.csv", index=False)
zone_kpi.to_csv(OUTPUT_DIR / "zone_kpi.csv", index=False)

print("Final dataset:", final_trips.shape)
print("Outputs written to:", OUTPUT_DIR)
print("\nFinal KPIs:")
display(final_kpis)

Final dataset: (3647543, 35)
Outputs written to: /content/output

Final KPIs:


,Metric,Value
0,Core Valid Trip Rate,95.39
1,Final Analytical Retention,95.06
2,Peak Demand Hour,18:00
3,Peak Hour Trips,231529
4,Median Trip Duration,14.28
5,P90 Trip Duration,33.23
6,Airport Median Duration,37.95
7,Non-Airport Median Duration,13.6
8,Highest Delay Zone,East Elmhurst
9,Highest Zone Delay Index,2.36


### Summary of Output Files

In [49]:
print('--- Summary for final_kpis.csv ---\n')
final_kpis_loaded = pd.read_csv('/content/output/final_kpis.csv')
print(f'Shape: {final_kpis_loaded.shape}')
print('\nInfo:')
display(final_kpis_loaded.info())
print('\nMissing values:')
display(final_kpis_loaded.isnull().sum())
print('\nFirst 5 rows:')
display(final_kpis_loaded.head())

--- Summary for final_kpis.csv ---

Shape: (10, 2)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Metric  10 non-null     object
 1   Value   10 non-null     object
dtypes: object(2)
memory usage: 292.0+ bytes


None


Missing values:


,0
Metric,0
Value,0



First 5 rows:


,Metric,Value
0,Core Valid Trip Rate,95.39
1,Final Analytical Retention,95.06
2,Peak Demand Hour,18:00
3,Peak Hour Trips,231529
4,Median Trip Duration,14.28


In [50]:
print('\n--- Summary for hourly_metrics.csv ---\n')
hourly_metrics_loaded = pd.read_csv('/content/output/hourly_metrics.csv')
print(f'Shape: {hourly_metrics_loaded.shape}')
print('\nInfo:')
display(hourly_metrics_loaded.info())
print('\nMissing values:')
display(hourly_metrics_loaded.isnull().sum())
print('\nFirst 5 rows:')
display(hourly_metrics_loaded.head())


--- Summary for hourly_metrics.csv ---

Shape: (24, 4)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pickup_hour      24 non-null     int64  
 1   trips            24 non-null     int64  
 2   median_duration  24 non-null     float64
 3   p90_duration     24 non-null     float64
dtypes: float64(2), int64(2)
memory usage: 900.0 bytes


None


Missing values:


,0
pickup_hour,0
trips,0
median_duration,0
p90_duration,0



First 5 rows:


,pickup_hour,trips,median_duration,p90_duration
0,0,131570,14.016667,29.000000
1,1,82971,13.333333,26.200000
2,2,55563,13.000000,25.033333
3,3,38566,12.916667,25.066667
4,4,30791,13.400000,26.766667


In [51]:
print('\n--- Summary for zone_kpi.csv ---\n')
zone_kpi_loaded = pd.read_csv('/content/output/zone_kpi.csv')
print(f'Shape: {zone_kpi_loaded.shape}')
print('\nInfo:')
display(zone_kpi_loaded.info())
print('\nMissing values:')
display(zone_kpi_loaded.isnull().sum())
print('\nFirst 5 rows:')
display(zone_kpi_loaded.head())


--- Summary for zone_kpi.csv ---

Shape: (79, 7)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   PULocationID     79 non-null     int64  
 1   PU_Borough       79 non-null     object 
 2   PU_Zone          79 non-null     object 
 3   trips            79 non-null     int64  
 4   median_duration  79 non-null     float64
 5   p90_duration     79 non-null     float64
 6   delay_index      79 non-null     float64
dtypes: float64(3), int64(2), object(2)
memory usage: 4.4+ KB


None


Missing values:


,0
PULocationID,0
PU_Borough,0
PU_Zone,0
trips,0
median_duration,0
p90_duration,0
delay_index,0



First 5 rows:


,PULocationID,PU_Borough,PU_Zone,trips,median_duration,p90_duration,delay_index
0,70,Queens,East Elmhurst,8196,32.133333,56.108333,2.362745
1,61,Brooklyn,Crown Heights North,4076,24.491667,43.325000,1.800858
2,181,Brooklyn,Park Slope,3459,24.033333,41.876667,1.767157
3,37,Brooklyn,Bushwick South,5320,23.858333,39.100000,1.754289
4,36,Brooklyn,Bushwick North,3130,23.350000,36.650000,1.716912


In [52]:
# Quality audit

quality_audit = pd.DataFrame({
    "Check": [
        "Raw rows",
        "Core validation failures",
        "Non-positive total amount",
        "Final analytical rows",
        "Duplicate rows after join",
        "Unknown pickup zone",
        "Unknown drop-off zone"
    ],
    "Value": [
        raw_rows,
        raw_rows - len(validated),
        (validated["total_amount"] <= 0).sum(),
        len(final_trips),
        final_trips.duplicated().sum(),
        (final_trips["PU_Zone"] == "Unknown").sum(),
        (final_trips["DO_Zone"] == "Unknown").sum()
    ]
})

quality_audit.to_csv(
    OUTPUT_DIR / "quality_audit.csv",
    index=False
)

display(quality_audit)

,Check,Value
0,Raw rows,3837248
1,Core validation failures,176773
2,Non-positive total amount,12932
3,Final analytical rows,3647543
4,Duplicate rows after join,0
5,Unknown pickup zone,3654
6,Unknown drop-off zone,3389


In [53]:
assert len(trips) > 0
assert len(validated) <= len(trips)
assert len(final_trips) <= len(validated)

assert final_trips.duplicated().sum() == 0
assert final_trips["PU_Zone"].isna().sum() == 0
assert final_trips["DO_Zone"].isna().sum() == 0

print(f"Pipeline completed successfully for {MONTH_ID}")
print(f"Raw rows: {len(trips)}")
print(f"Validated rows: {len(validated)}")
print(f"Final analytical rows: {len(final_trips)}")

Pipeline completed successfully for 2026-06
Raw rows: 3837248
Validated rows: 3660475
Final analytical rows: 3647543


In [54]:
# Save final analytical dataset
final_trips.to_parquet(
    OUTPUT_DIR / "final_trips.parquet",
    index=False
)

print("Saved:")
print(" - final_trips.parquet")
print(" - final_kpis.csv")
print(" - hourly_metrics.csv")
print(" - zone_kpi.csv")
print(" - quality_audit.csv")

Saved:
 - final_trips.parquet
 - final_kpis.csv
 - hourly_metrics.csv
 - zone_kpi.csv
 - quality_audit.csv


In [55]:
# =========================
# Pipeline Configuration
# =========================

YEAR = 2026
MONTH = 6

MONTH_ID = f"{YEAR}-{MONTH:02d}"

RAW_TRIPS = Path(f"/content/yellow_tripdata_{MONTH_ID}.parquet")
RAW_ZONES = Path("/content/taxi_zone_lookup.csv")

START_DATE = pd.Timestamp(YEAR, MONTH, 1)

if MONTH == 12:
    END_DATE = pd.Timestamp(YEAR + 1, 1, 1)
else:
    END_DATE = pd.Timestamp(YEAR, MONTH + 1, 1)

OUTPUT_DIR = Path(f"/content/output/{MONTH_ID}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Processing:", MONTH_ID)
print("Trip file:", RAW_TRIPS)
print("Period:", START_DATE.date(), "to", END_DATE.date())

Processing: 2026-06
Trip file: /content/yellow_tripdata_2026-06.parquet
Period: 2026-06-01 to 2026-07-01
